# 15 · Local pipeline: the backup plan end to end on one pair

This notebook runs `dtrack_local`, the backup plan that pulls both sides to this machine: `Warehouse` (Statement Execution API, inline vs external-link chunks, streaming to CSV), `WarehouseTarget` (Right counts and window pulls through the SQL warehouse, statistics computed locally), the `Landing` DuckDB, and the run itself via `open_run` and the stage functions / `run_all`, on one real pair and a small window, with the report and backup sent to an S3 scratch prefix.

It mirrors `tests/unit/dtrack_local/test_warehouse.py` and `tests/unit/dtrack_local/test_pipeline.py` (and test 6 of `docs/plans/prod_smoke_tests.md`), with the fake statement API, CSV anchor and moto replaced by the real warehouse, the real Left source and real S3.

## How to approach

- **Prerequisites**: kernel = the repo `.venv`; `uv sync --extra dbx` (+ `--extra athena` / `--extra ldap` if the pair's Left side needs it). `~/.databrickscfg` profile (`DTRACK_DBX_PROFILE`), SQL warehouse id (`DTRACK_DBX_WAREHOUSE_ID`), Left credentials `<macro>_USR/_PWD/_DSN` in the project `.env`, AWS credentials that can write `DTRACK_PROD_S3`.
- **Run order**: top to bottom. Sections 1-3 are small warehouse queries. Section 4 pulls one window of the Right table. Sections 6-7 run the whole plan for `PAIR` over `DAYS` days (Left pull + Right pull per window: keep `DAYS` small, 1-3). Only `WORKDIR` and `S3_SCRATCH/nb15-<tag>/` are written; Databricks is only read.
- **What to look at**: `status` DONE/PARTIAL with `dead == 0`, the findings per stage, `report/index.html`, the backup file count (no `_local/` uploaded), and the landing tables `left__<pair>` / `right__<pair>`.
- **Failure modes**: `preflight failed: <pair>.right ...` = wrong table/columns/grants; external-link download errors (SSL, proxy, 403) = presigned URLs blocked, set `DISPOSITION = "inline"` (<= 25 MiB per window); `statement FAILED` carries the warehouse error text; gate not passed = no day matches on both sides (row-count findings only, no Stage 2).

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")        # real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                                   # the single pair to run
TO_DATE = None                                                                      # None = yesterday; or "YYYY-MM-DD"
DAYS = 2                                                                            # small window: last DAYS days up to TO_DATE
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")    # report + backup land here, deleted at the end
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                          # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                        # SQL warehouse id
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")          # or "inline" if downloads are blocked
WORKDIR = Path.home() / ".local/plans/nbs/_work/15"                                 # runs/ and scratch files go here
GATE = "auto"                                                                       # Stage 1 gate: auto | manual | ratio:<x>
PUSH_SAMPLE = False                                                                 # True = filter the key sample in the warehouse too
KEEP = False                                                                        # True = keep scratch objects and WORKDIR (Notebook 14 can read the run)

In [ ]:
# Load the project .env first: the Left credentials (<macro>_USR / <macro>_PWD / <macro>_DSN)
# and the AWS / Databricks defaults live there. DTRACK_ENV_FILE overrides the search.
import copy
import json
import os
import uuid
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from dtrack_left.cli.config import load_config

load_dotenv(os.environ.get("DTRACK_ENV_FILE") or find_dotenv(usecwd=True), override=False)

# The real pairs config, validated exactly as `dtrack` validates it.
config = load_config(CONFIG_PATH)
print("pairs in config:", sorted(config["pairs"]))

# The check window: the last DAYS days up to TO (yesterday by default), like the prod smoke plan.
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()

# A private copy of the pair with its date range narrowed to the window; the config itself is untouched.
pair = copy.deepcopy(config["pairs"][PAIR])
pair["fromDate"], pair["toDate"] = FROM, TO
right_cfg = pair["right"]
TAG = f"{PAIR}-{uuid.uuid4().hex[:8]}"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"pair {PAIR}: window {FROM}..{TO}, scratch tag {TAG}")
print("right side:", json.dumps(right_cfg, indent=2, default=str))

SCRATCH_NAME = f"nb15-{TAG}"
scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"

## 1. Warehouse: one statement, polled, chunked

`Warehouse.execute` submits with `wait_timeout=30s` and `CONTINUE`, polls until the state leaves PENDING/RUNNING, and raises with the warehouse's error text otherwise. `chunks` follows `next_chunk_index` (inline `data_array` or presigned external links); `rows` flattens them.

In [ ]:
# One WorkspaceClient from ~/.databrickscfg and one Warehouse (Statement Execution API) on top of it.
# Nothing here needs Databricks Connect: every Right-side query goes through the SQL warehouse.
from dtrack_left.backends.databricks import workspace
from dtrack_local.warehouse import Warehouse

client = workspace(DBX_PROFILE)
warehouse = Warehouse(client, WAREHOUSE_ID, disposition=DISPOSITION)
print("signed in as", client.current_user.me().user_name, "on", client.config.host)
print("warehouse", WAREHOUSE_ID, "disposition", DISPOSITION, "->", warehouse.rows("SELECT current_catalog(), current_schema()"))

In [ ]:
from dtrack_right.backends.targets import SparkTarget

# The pair's real relation (same FROM as every Right query of the run).
relation = SparkTarget(None, right_cfg).relation()
print("relation:", relation)

# One real statement, inspected: id, final state, result columns, and how many chunks it came back in.
response = warehouse.execute(f"SELECT * FROM {relation} LIMIT 5")
print("statement", response.statement_id, "state", response.status.state)
print("columns:", [c.name for c in response.manifest.schema.columns])
chunks = list(warehouse.chunks(response))
print(f"{len(chunks)} chunk(s), rows per chunk: {[len(c) for c in chunks]}")
display(pd.DataFrame([row for chunk in chunks for row in chunk], columns=[c.name for c in response.manifest.schema.columns]))

## 2. Inline vs external links

The same grouped count is read with both dispositions: `INLINE` returns rows through the workspace host (<= 25 MiB), `EXTERNAL_LINKS` returns presigned URLs fetched by `download_chunk` without Databricks auth. Both must give identical rows; if presigned downloads are blocked on this machine the external-link run fails here.

In [ ]:
import time

from dtrack_local.warehouse import DISPOSITIONS, Warehouse
from dtrack_right.common.dates import bounds_where, day_expr

day = day_expr(right_cfg["date_col"], "spark", right_cfg.get("date_type"))
bounds = bounds_where(right_cfg["date_col"], dialect="spark", date_type=right_cfg.get("date_type"), from_date=FROM, to_date=TO)
count_sql = f"SELECT {day}, COUNT(*) FROM {relation} WHERE {SparkTarget(None, right_cfg)._where(bounds)} GROUP BY {day} ORDER BY 1"

by_disposition = {}
for disposition in sorted(DISPOSITIONS):
    started = time.monotonic()
    try:
        by_disposition[disposition] = Warehouse(client, WAREHOUSE_ID, disposition=disposition).rows(count_sql)
        print(f"{disposition:15} {len(by_disposition[disposition])} row(s) in {time.monotonic() - started:.1f}s")
    except Exception as exc:
        print(f"{disposition:15} FAILED: {type(exc).__name__}: {exc}")
if len(by_disposition) == 2:
    print("identical rows:", by_disposition["inline"] == by_disposition["external_links"])
display(pd.DataFrame(next(iter(by_disposition.values()), []), columns=["day", "rows"]))

## 3. Streaming a query to CSV, and the failure paths

`to_csv(sql, output, columns)` streams chunk by chunk through the same writer the Left pulls use and returns the data row count. A missing table raises `RuntimeError("statement FAILED: ...")`; a bad disposition or an empty warehouse id fails before any call.

In [ ]:
from dtrack_right.common.dates import partition_where, quote_ident

# The smallest day of the window, every mapped right column as text.
from dtrack_right.common.dates import canonical_date

day_counts = {canonical_date(d): int(n) for d, n in next(iter(by_disposition.values()), [])}
smallest = min(day_counts, key=day_counts.get) if day_counts else TO
columns = list(dict.fromkeys([right_cfg["date_col"], *pair["col_map"].values()]))
casts = ", ".join(f"CAST({quote_ident(c, 'spark')} AS STRING) AS {quote_ident(c, 'spark')}" for c in columns)
where = partition_where(right_cfg["date_col"], [smallest], dialect="spark", date_type=right_cfg.get("date_type"))
output = WORKDIR / "stream" / f"{PAIR}-{smallest}.csv"
pulled = warehouse.to_csv(f"SELECT {casts} FROM {relation} WHERE {SparkTarget(None, right_cfg)._where(where)}", output, columns)
print(f"{smallest}: pulled {pulled} row(s), counted {day_counts.get(smallest)} -> {output} ({output.stat().st_size} bytes)")
display(pd.read_csv(output, dtype=object, nrows=5))

# Failures carry the warehouse error text; bad settings fail early.
try:
    warehouse.rows("SELECT * FROM dtrack_nb_no_such_table_15")
except RuntimeError as exc:
    print("RuntimeError:", str(exc).splitlines()[0])
for kwargs in ({"warehouse_id": WAREHOUSE_ID, "disposition": "cloud"}, {"warehouse_id": ""}):
    try:
        Warehouse(client, **kwargs)
    except ValueError as exc:
        print("ValueError:", exc)

## 4. WarehouseTarget and the Landing DuckDB

`WarehouseTarget` counts with one `GROUP BY` on the warehouse, then for a window pulls `CAST(... AS STRING)` columns (date, keys, mapped columns) and computes the statistics locally. With a `Landing`, every pulled window is also kept in DuckDB (`right__<pair>`, one `dtrack_bucket` per window; re-landing a bucket replaces it). The manifest comes from real Left + Right counts.

In [ ]:
from dtrack_left.backends.sources import source_from_config
from dtrack_local.pipeline import Landing, WarehouseTarget
from dtrack_right.checks.row_check import evaluate_gate, reconcile_partitions, summarize_partitions
from dtrack_right.common.manifest import build_manifest, sample_plan

landing = Landing(WORKDIR / "landing" / "data.duckdb")
target = WarehouseTarget(warehouse, right_cfg, WORKDIR / "right", landing=landing, push_sample=PUSH_SAMPLE)
print("preflight:", {k: v for k, v in target.preflight().items() if k != "columns"})
right_counts = target.partition_counts(FROM, TO)
left_counts = source_from_config(pair["left"]).partition_counts(FROM, TO)

# Stage 1 by hand for this pair: reconcile, gate, manifest.
partitions = reconcile_partitions(left_counts, right_counts, pair.get("date_synonyms"))
document = {"partitions": partitions, "summary": summarize_partitions(partitions), "gate_decision": evaluate_gate(partitions, GATE)}
display(pd.DataFrame(partitions).T)
manifest = build_manifest(f"nb15-{TAG}", PAIR, pair, document)
window = manifest["windows"][0] if manifest["windows"] else None
print("windows:", [w["bucket"] for w in manifest["windows"]], "| sample:", manifest["sample"])
if window is None:
    raise RuntimeError("no day matched on both sides: widen DAYS or move TO_DATE")

In [ ]:
# The pull SQL (no md5 unless push_sample), the local statistics, and the landing table.
sql, selected = target.window_sql(manifest, window)
print("pull SQL:", sql[:800])
print("md5 sample pushed:", "md5(" in sql)
stats = target.window_statistics(manifest, window)
display(pd.DataFrame(stats).T)
print("landing:", landing.tables())

# Landing the same bucket again replaces it (count unchanged).
target.window_statistics(manifest, window)
print("landing after a second pull of the same bucket:", landing.tables())

# What push_sample would send to the warehouse: with the configured sample, or 10% as a labelled example.
plan = manifest["sample"] if manifest["sample"]["mode"] != "all" else sample_plan(0.1)
pushed = WarehouseTarget(warehouse, right_cfg, WORKDIR / "right_pushed", push_sample=True)
if manifest["right_key_cols"]:
    pushed_stats = pushed.window_statistics({**manifest, "sample": plan}, window)
    print("push_sample with", plan, "-> md5 in SQL:", "md5(" in pushed.window_sql({**manifest, "sample": plan}, window)[0])
    print({c: (stats[c]["n_total"], pushed_stats[c]["n_total"]) for c in stats}, "<- (all rows, pushed sample) n_total")

In [ ]:
import duckdb

# The landing file is plain DuckDB: raw text per bucket, ready for ad-hoc SQL.
with duckdb.connect(str(landing.path), read_only=True) as connection:
    table = f"right__{PAIR}"
    print(connection.execute(f'SELECT dtrack_bucket, count(*) FROM "{table}" GROUP BY 1').fetchall())
    display(connection.execute(f'SELECT * FROM "{table}" LIMIT 5').df())

## 5. open_run: the run folder

`open_run` takes the config (path or dict), a run id (default `<UTC stamp>-<config hash>`), the runs root, and the warehouse settings; by default it builds the `WarehouseTarget` factory itself with a landing DuckDB in `<run>/_local/`. The run folder has the same layout as an S3 run.

In [ ]:
from dtrack_local import pipeline

RUN_ID = f"nb15-{TAG}"
run = pipeline.open_run(
    {"pairs": {PAIR: pair}},
    run_id=RUN_ID,
    out=WORKDIR / "runs",
    profile=DBX_PROFILE,
    warehouse_id=WAREHOUSE_ID,
    disposition=DISPOSITION,
    push_sample=PUSH_SAMPLE,
    pairs=[PAIR],
)
print("run", run.run_id, "at", run.folder, "| store:", type(run.store).__name__, "| local:", run.local)
print("status before:", run.status())

# Preflight: Left SELECT ... WHERE 1=0 and Right DESCRIBE QUERY, nothing counted.
health = pipeline.preflight(run)
print(json.dumps({p: {s: {k: v for k, v in r.items() if k != "columns"} for s, r in sides.items()} for p, sides in health.items()}, indent=2, default=str))

## 6. Stage by stage: row_check, col_check, report

Stage 1 counts both sides, reconciles, gates and writes manifests. Stage 2 pulls every window on both sides, computes statistics and compares each unit. `report` writes `<run>/report/` and marks `_STATUS.json` complete. This is the slow part.

In [ ]:
# Stage 1.
stage_one = pipeline.row_check(run, gate=GATE)
print(json.dumps(stage_one, indent=2))
print("row_check closed:", run.store.exists("row_check/_CLOSED"), "| manifests:", run.store.list_keys("manifest/"))

In [ ]:
# Stage 2 (only when Stage 1 closed and released the pair).
stage_two = pipeline.col_check(run) if run.store.exists("row_check/_CLOSED") else None
print(json.dumps(stage_two, indent=2))
print("results:", len(run.store.list_keys("results/")), "| dead:", run.store.list_keys("dead/"))
for _key, failure in run.store.list_json("dead/"):
    print("DEAD", failure["unit_id"], failure["error"])

In [ ]:
# Report into <run>/report/ (no report_s3 here; section 7 publishes).
built = pipeline.report(run)
print(json.dumps({k: built["summary"][k] for k in ("status", "findings", "units_expected", "units_completed")}, indent=2))
print("files:", sorted(built["files"]))
print("status after:", run.status())
print("open:", (run.folder / "report" / "index.html").resolve().as_uri())

## 7. run_all on the same run: resume, publish, back up

`run_all` = preflight, Stage 1, Stage 2, report, then the optional report publish and S3 backup. On a run that already finished, the stages skip what is done, so the findings should be unchanged. The backup copies the run folder to `<backup_s3>/<run_id>/` without `_local/` (pulled data) unless `include_data=True`.

In [ ]:
import boto3

outcome = pipeline.run_all(run, gate=GATE, report_s3=f"{scratch}/report", backup_s3=f"{scratch}/runs")
print(json.dumps({k: outcome[k] for k in ("run_id", "status", "findings", "backup", "landing")}, indent=2, default=str))
print("same findings as section 6:", outcome["findings"] == built["summary"]["findings"])
print("dead units:", run.store.list_keys("dead/"), "| index.html:", (run.folder / "report" / "index.html").exists())

# What actually landed in S3: the backup (no _local/) and the published report with its pointer.
bucket, _, prefix = scratch.removeprefix("s3://").partition("/")
s3 = boto3.client("s3", endpoint_url=os.getenv("DTRACK_S3_ENDPOINT_URL") or None)
keys = [row["Key"].removeprefix(prefix + "/") for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/") for row in page.get("Contents") or []]
print(f"{len(keys)} object(s); _local uploaded:", any("/_local/" in k for k in keys))
print([k for k in keys if k.endswith(("index.html", "pointer.json", f"row_check/{PAIR}.json", "_STATUS.json"))])

## 8. The landing data of the run

Every window the run pulled, Left and Right, is in `<run>/_local/data.duckdb` as raw text. Comparing row counts per bucket on both sides is a quick sanity check on the windows that were compared.

In [ ]:
print("landing tables:", run.landing.tables())
with duckdb.connect(str(run.landing.path), read_only=True) as connection:
    names = [row[0] for row in connection.execute("SHOW TABLES").fetchall()]
    per_bucket = {
        name: dict(connection.execute(f'SELECT dtrack_bucket, count(*) FROM "{name}" GROUP BY 1 ORDER BY 1').fetchall())
        for name in names
    }
display(pd.DataFrame(per_bucket))

## 9. The same plan from the CLI

`python -m dtrack_local preflight|run` wraps `open_run` / `run_all` with the same options (`--pair`, `--out`, `--profile`, `--warehouse-id`, `--disposition`, `--push-sample`) and prints the outcome as JSON. Only `preflight` is called here (nothing counted); note it uses the pair's configured dates, not this notebook's window.

In [ ]:
from dtrack_local.cli import main

code = main([
    "preflight", "--config", CONFIG_PATH, "--pair", PAIR,
    "--run-id", f"nb15-cli-{TAG}", "--out", str(WORKDIR / "cli_runs"),
    "--profile", DBX_PROFILE, "--warehouse-id", WAREHOUSE_ID, "--disposition", DISPOSITION,
])
print("exit code:", code)
print("full run would be: python -m dtrack_local run --config", CONFIG_PATH, "--pair", PAIR, "--backup-s3", f"{scratch}/runs")

## Cleanup

Deletes `S3_SCRATCH/nb15-<tag>/` and `WORKDIR` (including the run folder) unless `KEEP = True`. Set `KEEP = True` first if you want Notebook 14 to read this run.

In [ ]:
# Delete everything this notebook wrote under the scratch prefix (and the local WORKDIR) unless KEEP.
import shutil

import boto3

scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"
if KEEP:
    print("KEEP=True: left", scratch, "and", WORKDIR)
else:
    bucket, _, prefix = scratch.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3", endpoint_url=os.getenv("DTRACK_S3_ENDPOINT_URL") or None)
    deleted = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/"):
        keys = [{"Key": row["Key"]} for row in page.get("Contents") or []]
        if keys:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": keys})
            deleted += len(keys)
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print(f"deleted {deleted} object(s) under {scratch} and {WORKDIR}")

## What to check

- Section 2: both dispositions return identical rows (or you know to use `inline`).
- Section 3: the streamed CSV row count equals the warehouse count for that day.
- Section 4: the pull SQL has no `md5(` unless `PUSH_SAMPLE`; landing count is stable after re-pulling a bucket.
- Sections 5-6: preflight ok on both sides; Stage 1 gate passed; Stage 2 `dead == 0`; status DONE (or PARTIAL with a reason).
- Section 7: `run_all` findings equal the stage-by-stage ones; backup has files, no `_local/`; `report/latest/pointer.json` exists.
- Section 8: left and right landing counts agree per bucket for compared windows.